# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa – Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset is sourced via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load dataset metadata and gain insight into its context using `mlcroissant`. The Croissant metadata provides structured access to dataset content, including available record sets, fields, and columns.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}\n{metadata.description}\n")

## 2. Data Overview

Display all available record sets, their `@id`s, associated fields (columns), and descriptive summaries. All structural entities are referenced by their `@id` to ensure traceability and reproducibility.

In [ ]:
# List record sets, fields, and columns by `@id`
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - RecordSet name: {rs.name} (id: {rs.id})")
    print("    Fields/columns:")
    for field in rs.fields:
        print(f"      * Field: {field.name} (id: {field.id}, dtype: {field.data_type})")
    print()


## 3. Data Extraction

Now we'll extract data from each record set into a pandas DataFrame for analysis. Here, we use the record set and field `@id`s reported above.

In [ ]:
# Extract all dataframes keyed by their record set @id
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set in dataset.record_sets:
    records = list(dataset.records(record_set=record_set.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set.id] = df
        print(f"Loaded dataframe for record set '{record_set.name}' (@id: {record_set.id}): {df.shape[0]} rows x {df.shape[1]} columns")

# For demonstration, pick the main survey responses record set (by inspection; adapt id if needed):
main_rs_id = record_set_ids[0]  # Update this index if you want a different recordset
print(f"\nExample columns for main record set (@id: {main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's process a numeric field (e.g., the GAD-7 score or a similar mental health assessment). All references use the relevant field `@id`. We'll filter out records with high scores, normalize, and explore basic grouping.

Replace the `numeric_field_id` and `group_field_id` values below if your schema uses different `@id`s for GAD-7, PHQ-9, or PSQ scores.

In [ ]:
# Set the field @id for GAD-7 score (update as necessary based on schema fields)
numeric_field_id = None
group_field_id = None

# Attempt to auto-select a likely numeric field based on field names
for field in dataset.record_sets[0].fields:
    if 'gad' in field.name.lower() or 'phq' in field.name.lower() or 'psq' in field.name.lower():
        numeric_field_id = field.id
    if group_field_id is None and ('gender' in field.name.lower() or 'education' in field.name.lower()):
        group_field_id = field.id

if numeric_field_id is None:
    raise ValueError("Could not auto-detect a numeric field (e.g., GAD-7/PHQ-9/PSQ). Please look up field IDs in the previous cell.")

df = dataframes[main_rs_id]
print(f"Using numeric field for EDA: {numeric_field_id}")
threshold = 10

if numeric_field_id not in df.columns:
    raise ValueError(f"Field {numeric_field_id} not in columns. Check spelling and field @id.")
filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold}:")
print(filtered_df[[numeric_field_id]].head())

filtered_df = filtered_df.copy()
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(f"mean_{numeric_field_id}")
    print(f"\nGrouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of mental health scores (e.g., GAD-7/PHQ-9/PSQ). Adjust field `@id` as needed. Example: Histogram distribution and group-wise boxplot.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

- We loaded FAIR² Mental Health Survey data using the Croissant standard and `mlcroissant`.
- The survey includes rich demographic and mental health assessment data (e.g., GAD-7, PHQ-9, PSQ scores).
- Using record set/field `@id`s ensures robust data processing and reproducibility.
- This notebook provides an extensible template for further analysis, modeling, and FAIR cross-dataset integration.